In [3]:
import os
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import json as json_lib 
from pub import Weak, Random, Real

def main():
    op_dir = '../data/A123/'
    ip_f = '../init/ASSIST0910/skill_builder_data_corrected_collapsed.csv'
    seed = 123
    os.makedirs(op_dir, exist_ok=True)

    print("="*10 + " 1 Data Loading " + "="*10)
    cols = ['order_id', 'user_id', 'problem_id', 'skill_id', 'correct']
    full_df = pd.read_csv(ip_f, encoding='latin1', usecols=cols)
    full_df.dropna(inplace=True)
    df = full_df.drop_duplicates(subset=['user_id', 'problem_id'], keep='first')
    df = df.groupby('user_id').filter(lambda g: len(g) >= 15).copy()

    def k2list(x):
        if isinstance(x, float): 
            return [int(x)]
        return [int(i) for i in str(x).split('_')]

    s_all = df['user_id'].unique()
    e_all = df['problem_id'].unique()
    df['skill_list'] = df['skill_id'].apply(k2list)
    k_all = set([k for ks in df['skill_list'] for k in ks])
    
    n, m, k = len(s_all), len(e_all), len(k_all)
    print(f"{n} users, {m} problems, {k} skills.")

    s2idx = {int(uid): i for i, uid in enumerate(s_all)}
    e2idx = {int(pid): i for i, pid in enumerate(e_all)}
    k2idx = {int(sid): i for i, sid in enumerate(k_all)}

    n, m, k = len(s2idx), len(e2idx), len(k2idx)
    print(f"Verify: {n} users, {m} problems, {k} skills.")

    print("\n" + "="*10 + " 2 ID Mapping " + "="*10)
    df['user_id'] = df['user_id'].map(s2idx)
    df['problem_id'] = df['problem_id'].map(e2idx)
    df['skill_list'] = df['skill_list'].apply(lambda ks: [k2idx[s] for s in ks])

    p2k = df.groupby('problem_id')['skill_list'].first().to_dict()
    p2k_to_save = pd.DataFrame(p2k.items(), columns=['problem_id', 'skill_ids'])
    p2k_to_save.to_csv(os.path.join(op_dir, 'p2k.csv'), index=False)
    
    n_records = len(df)
    avg_k_per_e = np.mean([len(ks) for ks in p2k.values()])
    avg_r_per_s = n_records / n
    n_correct = df[df['correct'] == 1].shape[0]
    n_incorrect = df[df['correct'] == 0].shape[0]
    avg_c_per_s = df.groupby('user_id')['skill_list'].apply(
        lambda x: len(set([item for sublist in x for item in sublist]))
    ).mean()

    print("\n## Dataset Statistics")
    print(f"- records: {n_records}")
    print(f"- students: {n}")
    print(f"- questions: {m}")
    print(f"- knowledge concepts: {k}")
    print(f"- concepts per exercise: {avg_k_per_e:.2f}")
    print(f"- records per student: {avg_r_per_s:.2f}")
    print(f"- avg concepts per student: {avg_c_per_s:.2f}")
    print(f"- correct records / incorrect records: {n_correct} / {n_incorrect}")


    meta = {'n': n, 'm': m, 'k': k, 'avg_k': avg_c_per_s, 's2idx': s2idx, 'e2idx': e2idx, 'k2idx': k2idx}
    with open(os.path.join(op_dir, "meta.json"), 'w') as f:
        json.dump(meta, f)

    print("\n" + "="*10 + " 3 kc_counts " + "="*10)
    
    sp_df = df[['user_id', 'problem_id', 'correct', 'order_id']].copy()
    sp_df.sort_values(by=['user_id', 'order_id'], inplace=True)
    sp_df['skill_ids'] = sp_df['problem_id'].map(p2k)
    
    kc_counts_series = pd.Series(index=sp_df.index, dtype=str)

    for user_id, group in tqdm(sp_df.groupby('user_id'), "Generating kc_counts"):
        kc_counter = np.zeros(k, dtype=np.int16)
        
        for index, row in group.iterrows():
            kc_counts_series.at[index] = json_lib.dumps(kc_counter.tolist())
            
            k_practiced = row['skill_ids']
            if k_practiced:
                kc_counter[k_practiced] += 1
    
    sp_df['kc_counts'] = kc_counts_series
    sp_df.drop(columns=['skill_ids'], inplace=True)
    print("kc_counts calculated successfully.")

    weak_splitter = Weak(op_dir, seed)
    weak_splitter.split_and_save(sp_df, p2k)

    random_splitter = Random(op_dir, n_folds=5, seed=seed)
    random_splitter.split_and_save(sp_df, p2k)
    
    real_splitter = Real(op_dir=op_dir)
    real_splitter.split_and_save(sp_df, p2k)
    
    print("All splitting done.")

if __name__ == '__main__':
    main()

========== 1 Data Loading ==========


C:\Users\jy_18\AppData\Local\Temp\ipykernel_10368\952773577.py:17: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  full_df = pd.read_csv(ip_f, encoding='latin1', usecols=cols)


2493 users, 17676 problems, 123 skills.
Verify: 2493 users, 17676 problems, 123 skills.

========== 2 ID Mapping ==========

## Dataset Statistics
- records: 267423
- students: 2493
- questions: 17676
- knowledge concepts: 123
- concepts per exercise: 1.20
- records per student: 107.27
- avg concepts per student: 15.38
- correct records / incorrect records: 175890 / 91533

========== 3 kc_counts ==========


Generating kc_counts: 100%|██████████| 2493/2493 [00:15<00:00, 157.78it/s]


kc_counts calculated successfully.

========== 1 Weak-Split (Corrected) ==========


...

Weak response proportion: 0.628
Weak Split Done. tv: 214874, test: 52549

========== 2 Random 5-Fold Split ==========


Random Split: 100%|██████████| 2493/2493 [00:01<00:00, 1625.32it/s]


Random 5-Fold Split Done. Avg Weak response proportion: 0.028

========== 3 Real-Scenario (Time-Series) Split ==========


Time-Series Split: 100%|██████████| 2493/2493 [00:00<00:00, 21933.17it/s]


Weak response proportion: 0.333
Real Split Done. Train: 159500, Val: 52549, Test: 55374
All splitting done.
